# Rollout Collection: Workstream 1 scaffold vs. the real gemma-4-31b-it-qat-w4a16-ct

Run our actual `submission/` scaffold (not the sample submission) against the real 31B served on this notebook's GPU(s).

**Runs 1-7** iterated on `eval_config.yaml` and four root-cause agent bugs (`search_similar_code`'s context-overflow risk, pytest output-truncation blindness, the graph's async blind spot, `skip_summarization: true`'s wasted-turn tax, and a real `max_output_tokens` context-overflow crash), converging on a config that scored 20% resolved with ~9.8h extrapolated total time across 10 held-out tasks -- submitted as `56752702`.

**That submission crashed anyway** ("unhandled error while rerunning your code"), the same generic failure as the previous attempt. Comparing all three real submissions: the only one that *didn't* crash was also the slowest/least-efficient one (most likely to have been cut off by the 12h wall before finishing all ~120 hidden tasks) -- consistent with a crash reachable only once the agent is fast enough to get deep into the hidden task list, not something our 10-task sample would ever surface.

**Methodology gap found**: every run 1-7 called `evaluator.evaluate_task()` directly in a manual Python loop -- never once `Evaluator(config).run()`, the actual top-level method `swegemma` exposes and almost certainly what the real scorer calls internally. `run()` exercises concurrency handling and result aggregation (`save_results`/`task_results.jsonl`/`summary.json`) that the manual loop never touched. Also never tested: any `httpx` task (one of the 4 official repos, completely uncovered by runs 1-7).

**Run 8** (this run): switches to `Evaluator(config).run()` with `concurrency=4` (plausible for 4x L4 GPUs, and specifically exercises the concurrent code path none of runs 1-7 touched) and `display_mode='quiet'` (avoids the live-dashboard rendering as an extra untested variable while isolating the concurrency path itself), against a broader 22-task batch spanning all 4 official repos including `httpx` for the first time. Goal: surface any crash locally (with a real traceback) before ever risking another submission slot.

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Our Submission Scaffold and Task Selection

The submission dataset (`noobhunterjay/gemma4-agent-submission`) mirrors the git-tracked `submission/` directory in the project repo exactly, including the graph-tool-usage and eval_config.yaml fixes made after submission 56682704 timed out -- NOT the competition's `sample_submission/`. Rather than guessing the exact Kaggle mount-path convention (it has changed between competitions before), we search for it by content.

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Find our submission dataset under /kaggle/input by content (agent.yaml naming ours
# 'swe_scaffold_v1'), rather than assuming a specific mount-path convention.
candidates = [
    p.parent for p in Path('/kaggle/input').rglob('agent.yaml')
    if 'swe_scaffold_v1' in p.read_text(encoding='utf-8')
]
assert candidates, 'Could not find our submission dataset under /kaggle/input -- check dataset_sources in kernel-metadata.json'
OUR_SUBMISSION_SRC = candidates[0]
print(f'Found our submission scaffold at: {OUR_SUBMISSION_SRC}')

AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(OUR_SUBMISSION_SRC, AGENT_DIR)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
by_id = {t.instance_id: t for t in all_tasks}

# Run 8: broader 22-task batch spanning all 4 official repos -- mostly fresh rich
# tasks (not reused from runs 1-7), a fresh fastapi task plus some repeats for
# continuity, all 3 resolvable requests tasks, and the ONE existing httpx task
# (never tested in any prior run; locally 0/1 resolvable via gold patch due to a
# known local pytest-httpbin dependency gap, but that doesn't mean the AGENT itself
# can't run against it -- included specifically for crash-surface coverage, not an
# expectation it resolves).
SMOKE_TEST_IDS = [
    # rich: fresh instance_ids not used in runs 1-7
    'rich_3935', 'rich_3905', 'rich_3930', 'rich_3180', 'rich_3777', 'rich_3468',
    'rich_3772', 'rich_3675', 'rich_3676', 'rich_3518', 'rich_3535', 'rich_3521',
    # fastapi: one fresh instance_id plus repeats for continuity
    'fastapi_15800', 'fastapi_15661', 'fastapi_15588', 'fastapi_14851', 'fastapi_14873', 'fastapi_14372',
    # requests: all 3 locally-resolvable instance_ids
    'requests_7427', 'requests_7315', 'requests_7309',
    # httpx: the only official-pool instance_id, never tested before
    'httpx_3672',
]
SAMPLE_TASKS = [by_id[i] for i in SMOKE_TEST_IDS if i in by_id]

print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(all_tasks)} total tasks; running validation batch on {len(SAMPLE_TASKS)}:')
for t in SAMPLE_TASKS:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Start vLLM Server (real `gemma-4-31b-it-qat-w4a16-ct`)

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters (none yet -- Workstream 3)
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4. Rollout Collection: run the agent, verify, keep only resolved trajectories

Mirrors `scripts/collect_rollouts.py`'s logic (same two-phase `Evaluator.evaluate_task` the real scorer uses), adapted for the notebook's `subprocess` sandbox instead of local Docker. Every attempt (resolved or not) is recorded; the ATIF trace for each is saved so resolved ones can be turned into SFT training data afterward.

In [ ]:
import asyncio
import concurrent.futures
import json
import time
import traceback
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_fn, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn(**kwargs))).result()
    return asyncio.run(coro_fn(**kwargs))


eval_config_file = AGENT_DIR / 'eval_config.yaml'
budget_overrides = {}
if eval_config_file.exists():
    raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
    eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)
    if 'timeout_seconds' in eval_section:
        budget_overrides['timeout_seconds'] = int(eval_section['timeout_seconds'])
    if 'max_tool_calls' in eval_section:
        budget_overrides['max_tool_calls'] = int(eval_section['max_tool_calls'])
    if 'max_time_minutes' in eval_section:
        budget_overrides['max_time_minutes'] = float(eval_section['max_time_minutes'])
    turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
    if turns_raw is not None:
        budget_overrides['max_turns'] = int(turns_raw)
    print(f'Found eval_config.yaml -- overrides: {budget_overrides}')
else:
    print('No eval_config.yaml found -- using pure harness defaults.')

RESULTS_DIR = WORKING_DIR / 'rollouts'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    task_ids=[t.instance_id for t in SAMPLE_TASKS],
    concurrency=4,
    display_mode='quiet',
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
    **budget_overrides,
)
print(f'Effective per-task budget: time_minutes={eval_config.budget.time_minutes}, tool_calls={eval_config.budget.tool_calls}, turns={eval_config.budget.turns}, command_timeout_seconds={eval_config.harness.command_timeout_seconds}')
print(f'concurrency={eval_config.concurrency}, display_mode={eval_config.display_mode}')

evaluator = Evaluator(eval_config)
start = time.perf_counter()
try:
    # The REAL top-level method (runs 1-7 only ever called evaluate_task() directly
    # in a manual loop, bypassing this entirely) -- exercises concurrency handling
    # and result aggregation (save_results -> task_results.jsonl/summary.json) the
    # same way the real scorer almost certainly does.
    eval_result = run_sync(evaluator.run)
    print(f'\nEvaluator.run() completed in {time.perf_counter() - start:.1f}s')
except Exception:
    print(f'\nEvaluator.run() RAISED after {time.perf_counter() - start:.1f}s -- this is the kind of crash we could never see in a real submission:')
    traceback.print_exc()
    raise

print(f'\n{eval_result.resolved}/{eval_result.total} resolved ({eval_result.resolution_rate:.1%})')
for repo, stats in sorted(eval_result.by_repo().items()):
    print(f'  {repo}: {stats}')
print(f'\nResults written to {RESULTS_DIR} (task_results.jsonl, summary.json, patches/, traces/)')